### IMPORT LIBLARY

In [ ]:
import os
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

------

### 1. Persiapan Lingkungan dan Import Library
Tahap awal dalam proyek ini adalah mempersiapkan *library* yang diperlukan untuk mendukung proses analisis citra dan pemodelan *machine learning*. *Library* yang digunakan meliputi:

* **Pemrosesan Citra (`cv2`, `skimage`):** Digunakan untuk membaca, memanipulasi, dan mengekstraksi fitur tekstur GLCM (*Gray Level Co-Occurrence Matrix*) dari citra pisang.
* **Manipulasi Data (`numpy`, `pandas`):** Digunakan untuk mengelola struktur data citra dan *dataset* fitur agar siap diolah oleh model.
* **Visualisasi (`matplotlib`, `seaborn`):** Digunakan untuk menampilkan hasil analisis, grafik perbandingan model, dan matriks kebingungan (*confusion matrix*).
* **Pemodelan & Evaluasi (`sklearn`, `scipy`):** Digunakan untuk membangun model *Machine Learning* (KNN, SVM, Random Forest), melakukan pembagian data (*train-test split*), serta menghitung metrik evaluasi performa model seperti akurasi, presisi, dan F1-Score.

-------

### PEMUAT DATASET

In [ ]:
data = []
labels = []
file_name = []

for sub_folder in os.listdir("./Banana Ripeness Classification Dataset"):
    sub_folder_path = os.path.join("./Banana Ripeness Classification Dataset", sub_folder)
    
    if os.path.isdir(sub_folder_path):
        sub_folder_files = os.listdir(sub_folder_path)
        for i, filename in enumerate(sub_folder_files):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(sub_folder_path, filename)
                img = cv.imread(img_path)
                
                if img is not None:
                    img = img.astype(np.uint8)
                    data.append(img)
                    labels.append(sub_folder)
                    file_name.append(filename)
        
data = np.array(data, dtype=object)
labels = np.array(labels)
print(f"Berhasil memuat {len(data)} gambar dari folder dataset.")

------

### 2. Pemuatan Dataset (Dataset Loading)
Tahap ini bertujuan untuk mengimpor seluruh citra dari direktori dataset lokal ke dalam memori kerja Python. Langkah-langkah yang dilakukan adalah:

* **Iterasi Folder:** Kode melakukan *looping* melalui sub-folder yang ada di dalam direktori `./Banana Ripeness Classification Dataset`. Setiap sub-folder diasumsikan mewakili label kategori kematangan pisang tertentu.
* **Filter File:** Sistem secara otomatis memfilter file agar hanya memproses file dengan ekstensi gambar yang valid (`.png`, `.jpg`, `.jpeg`), sehingga menghindari error akibat file sistem lain (seperti file tersembunyi).
* **Pembacaan Citra:** Fungsi `cv.imread` digunakan untuk membaca data piksel dari gambar dan menyimpannya ke dalam list `data`.
* **Sinkronisasi Label:** Nama sub-folder diambil dan disimpan ke dalam list `labels` sebagai label klasifikasi (ground truth), sedangkan nama file asli disimpan ke dalam list `file_name` untuk memudahkan pelacakan data selama proses pengujian.
* **Konversi ke Array:** Terakhir, data dikonversi ke dalam format `numpy.array` agar lebih efisien untuk diolah dalam proses komputasi vektor pada tahap ekstraksi fitur dan pelatihan model.

-------

### DATA EKPLORATIF

In [ ]:
unique_labels, counts = np.unique(labels, return_counts=True)
plt.figure(figsize=(8, 5))
sns.barplot(x=unique_labels, y=counts, hue=unique_labels, palette='viridis', legend=False)

plt.title('Grafik Distribusi Jumlah Gambar per Kelas', fontsize=14)
plt.xlabel('Label Kelas', fontsize=12)
plt.ylabel('Jumlah Gambar', fontsize=12)
plt.show()

print("Menampilkan 5 sampel gambar dari dataset:")
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
random_indices = np.random.choice(len(data), 5, replace=False)

for i, idx in enumerate(random_indices):
    img_array = np.array(data[idx], dtype=np.uint8)
    
    img_rgb = cv.cvtColor(img_array, cv.COLOR_BGR2RGB) if len(img_array.shape) == 3 else img_array
    
    axes[i].imshow(img_rgb)
    axes[i].set_title(labels[idx])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

--------

### 3. Analisis Data Eksploratif (EDA)
Sebelum melakukan pemodelan, sangat penting untuk memahami karakteristik dataset yang digunakan. Tahap ini terbagi menjadi dua bagian:

* **Distribusi Jumlah Gambar per Kelas:**
  Menggunakan `sns.barplot`, kita memvisualisasikan jumlah sampel untuk setiap kategori kematangan pisang. Langkah ini krusial untuk mendeteksi apakah terjadi **ketidakseimbangan data (*class imbalance*)**, yaitu kondisi di mana jumlah sampel antar kelas tidak merata yang dapat memengaruhi bias model saat proses pelatihan.

* **Visualisasi Sampel Gambar:**
  Bagian ini menampilkan 5 sampel gambar secara acak dari dataset menggunakan `np.random.choice`. 
  * Karena `OpenCV` membaca gambar dalam format **BGR**, dilakukan konversi ke format **RGB** agar warna gambar yang muncul pada `matplotlib` sesuai dengan aslinya.
  * Tujuannya adalah untuk memberikan gambaran visual mengenai variasi kualitas gambar dan memastikan bahwa setiap citra telah dimuat dengan benar ke dalam sistem sebelum masuk ke tahap ekstraksi fitur.

  --------

### AUGMENTASI DATA

In [ ]:
data_augmented = []
labels_augmented = []
print("Data sebelum augmentasi: ", len(data))

------

### 4. Augmentasi Data (Data Augmentation)
Pada tahap ini, kita melakukan teknik augmentasi untuk memperbanyak jumlah dataset secara buatan (*artificial expansion*). Tujuan utama dari langkah ini adalah:

* **Mengatasi Keterbatasan Data:** Memperbanyak variasi data agar model memiliki lebih banyak contoh untuk dipelajari, yang secara langsung dapat membantu mengurangi risiko *overfitting*.
* **Invariansi Model:** Dengan memanipulasi gambar (seperti rotasi, *flipping*, atau perubahan kecerahan—tergantung fungsi yang kamu gunakan), model diajarkan untuk mengenali objek pisang dari berbagai sudut dan kondisi pencahayaan yang berbeda.
* **Validasi Awal:** Baris `print("Data sebelum augmentasi: ", len(data))` berfungsi sebagai kontrol untuk mencatat ukuran *dataset* awal sebelum proses duplikasi atau modifikasi diterapkan, sehingga kita bisa membandingkan secara transparan berapa banyak data tambahan yang berhasil dihasilkan.

-------

### PRA-PROSESAN CITRA

In [ ]:
def prepro3_visual(image):
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY) if len(image.shape) == 3 else image.copy()
    median_blur = cv.medianBlur(gray, 5)
    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5))
    opening_result = cv.morphologyEx(median_blur, cv.MORPH_OPEN, kernel)
    return gray, median_blur, opening_result

dataPreprocessed = []

for i in range(len(data)):
    if data[i] is None: continue

    img_array = np.array(data[i], dtype=np.uint8)
    img_resized = cv.resize(img_array, (256, 256))
    
    _, _, final_prep = prepro3_visual(img_resized)
    dataPreprocessed.append(final_prep)

dataPreprocessed = np.array(dataPreprocessed)

print("Menampilkan Grafik Hasil Preprocessing (Sampel Acak):")
sample_idx = np.random.choice(len(dataPreprocessed), 3, replace=False)

for idx in sample_idx:
    img_array = np.array(data[idx], dtype=np.uint8)
    img_resized = cv.resize(img_array, (256, 256))
    img_rgb = cv.cvtColor(img_resized, cv.COLOR_BGR2RGB) if len(img_resized.shape) == 3 else img_resized
    
    gray, median, opening = prepro3_visual(img_resized)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img_rgb); axes[0].set_title("1. Asli"); axes[0].axis('off')
    axes[1].imshow(gray, cmap='gray'); axes[1].set_title("2. Grayscale"); axes[1].axis('off')
    axes[2].imshow(median, cmap='gray'); axes[2].set_title("3. Median Filter"); axes[2].axis('off')
    axes[3].imshow(opening, cmap='gray'); axes[3].set_title("4. Morfologi Opening"); axes[3].axis('off')
    plt.tight_layout()
    plt.show()

print("Preprocessing untuk seluruh dataset selesai. Dimensi:", dataPreprocessed.shape)

--------

### 5. Pra-pemrosesan Citra (Image Preprocessing Pipeline)
Tahap ini adalah langkah krusial untuk menstandarisasi input citra agar model dapat mengekstraksi fitur tekstur dengan lebih stabil. Proses ini terdiri dari beberapa tahapan:

* **Resizing:** Seluruh citra diubah ukurannya menjadi (256x256) piksel. Hal ini wajib dilakukan agar dimensi data seragam, yang akan mencegah error saat perhitungan matriks GLCM nantinya.
* **Transformasi Grayscale:** Mengonversi citra BGR ke skala abu-abu (*grayscale*) untuk mereduksi kompleksitas data dari 3 channel warna menjadi 1 channel intensitas cahaya.
* **Median Filtering:** Mengaplikasikan `cv.medianBlur` dengan kernel 5x5. Teknik ini sangat efektif untuk menghilangkan *noise* jenis *salt-and-pepper* tanpa mengaburkan tepi objek yang krusial untuk analisis tekstur.
* **Morfologi Opening:** Menggunakan operasi *Erosion* diikuti *Dilation* dengan elemen struktur elips. Proses ini bertujuan untuk menghapus detail kecil yang tidak diinginkan pada latar belakang dan mempertegas tekstur kulit pisang.

Visualisasi di bawah ini menampilkan perbandingan perubahan citra dari fase asli hingga hasil akhir *preprocessing* yang akan digunakan sebagai input model klasifikasi.

---------

### EKSTRASI FITUR GLCM

In [ ]:
def glcm(image, derajat):
    angles = [0] if derajat == 0 else [np.pi/4] if derajat == 45 else [np.pi/2] if derajat == 90 else [3*np.pi/4]
    return graycomatrix(image, [1], angles, 256, symmetric=True, normed=True)

def correlation(m): return graycoprops(m, 'correlation')[0, 0]
def dissimilarity(m): return graycoprops(m, 'dissimilarity')[0, 0]
def homogenity(m): return graycoprops(m, 'homogeneity')[0, 0]
def contrast(m): return graycoprops(m, 'contrast')[0, 0]
def ASM(m): return graycoprops(m, 'ASM')[0, 0]
def energy(m): return graycoprops(m, 'energy')[0, 0]
def entropyGlcm(m): return entropy(m.ravel())

--------

### 6. Ekstraksi Fitur Tekstur (GLCM)
Untuk merepresentasikan karakteristik tekstur kulit pisang dalam bentuk numerik, kita menggunakan metode GLCM. GLCM menghitung seberapa sering pasangan piksel dengan nilai intensitas tertentu muncul dalam jarak dan sudut orientasi yang ditentukan.

* **Fungsi `glcm`:** Fungsi ini berfungsi untuk membangun matriks GLCM berdasarkan derajat orientasi yang dipilih (0°, 45°, 90°, atau 135°). Penggunaan `symmetric=True` memastikan hubungan antar piksel bersifat dua arah, sementara `normed=True` menormalisasi nilai matriks sehingga probabilitas kemunculan piksel lebih mudah dibandingkan antar kelas.

* **Fungsi Ekstraksi Properti:** Setelah matriks GLCM terbentuk, kita mengekstrak parameter statistik utama yang merepresentasikan karakteristik tekstur permukaan pisang:
    * **Correlation:** Mengukur ketergantungan antar piksel (tekstur linear).
    * **Dissimilarity & Contrast:** Mengukur variasi lokal; nilai tinggi menunjukkan perbedaan kontras yang tajam (misalnya bercak hitam pada pisang matang).
    * **Homogeneity:** Mengukur keseragaman distribusi piksel; nilai tinggi menunjukkan permukaan yang halus.
    * **ASM (*Angular Second Moment*) & Energy:** Mengukur keteraturan tekstur.
    * **Entropy:** Mengukur tingkat ketidakteraturan atau kompleksitas tekstur pada citra.

Metode ini sangat efektif untuk mengklasifikasikan tingkat kematangan pisang karena perubahan warna kulit pisang dari mentah ke matang sangat berkorelasi dengan perubahan tekstur dan distribusi intensitas piksel.

---------

### PENYUSUNAN TABEL FITUR

In [ ]:
fitur = {k: [] for k in ['D0','D45','D90','D135','K0','K45','K90','K135','Dis0','Dis45','Dis90','Dis135',
                         'H0','H45','H90','H135','E0','E45','E90','E135','A0','A45','A90','A135',
                         'ER0','ER45','ER90','ER135','C0','C45','C90','C135']}

for img in dataPreprocessed:
    m0, m45, m90, m135 = glcm(img, 0), glcm(img, 45), glcm(img, 90), glcm(img, 135)
    
    for m, ang in zip([m0, m45, m90, m135], ['0', '45', '90', '135']):
        fitur[f'K{ang}'].append(contrast(m))
        fitur[f'Dis{ang}'].append(dissimilarity(m))
        fitur[f'H{ang}'].append(homogenity(m))
        fitur[f'E{ang}'].append(entropyGlcm(m))
        fitur[f'A{ang}'].append(ASM(m))
        fitur[f'ER{ang}'].append(energy(m))
        fitur[f'C{ang}'].append(correlation(m))

dataTable = {'Filename': file_name, 'Label': labels}
mapping = {'Contrast':'K', 'Dissimilarity':'Dis', 'Homogeneity':'H', 'Entropy':'E', 'ASM':'A', 'Energy':'ER', 'Correlation':'C'}
for name, prefix in mapping.items():
    for ang in ['0', '45', '90', '135']:
        dataTable[f'{name}{ang}'] = fitur[f'{prefix}{ang}']

df = pd.DataFrame(dataTable)
df.to_csv('hasil_ekstraksi_1.csv', index=False)

hasilEkstrak = pd.read_csv('hasil_ekstraksi_1.csv')
print("Tabel Hasil Ekstraksi Fitur GLCM (5 Baris Pertama):")
display(hasilEkstrak.head())

--------

### 7. Penyusunan Tabel Fitur (Feature Dataset Assembly)
Setelah nilai statistik tekstur (GLCM) berhasil diekstraksi, tahap selanjutnya adalah menyusunnya ke dalam struktur data yang terorganisir agar dapat digunakan dalam proses klasifikasi.

* **Inisialisasi Struktur Data:** Kita menggunakan *dictionary* `fitur` dengan *key* yang merepresentasikan setiap kombinasi properti (seperti Kontras, Energi, dll) dan sudut orientasi (0°, 45°, 90°, 135°).
* **Iterasi Ekstraksi:** *Looping* dilakukan pada seluruh citra yang telah diproses (`dataPreprocessed`), di mana setiap citra dihitung nilai GLCM-nya untuk keempat sudut tersebut. Hal ini memastikan model memiliki wawasan tekstur yang **invarian terhadap rotasi**, artinya model tetap mampu mengenali pola pisang meski sudut pengambilan fotonya berbeda-beda.
* **Integrasi dengan Label:** Kita menggabungkan fitur numerik tersebut dengan informasi metadata (`Filename` dan `Label`) ke dalam satu objek `dataTable`. Hal ini penting agar model dapat mengenali hubungan antara fitur tekstur dan kategori kematangan pisang yang sebenarnya (ground truth).
* **Ekspor Data:** Data yang telah tersusun rapi dalam `DataFrame` kemudian disimpan ke dalam file `hasil_ekstraksi_1.csv`. Langkah ini penting untuk **audit data**, di mana kita dapat memeriksa kembali nilai-nilai tekstur yang telah diekstraksi tanpa harus melakukan komputasi ulang dari awal.

----------

### SELEKSI FITUR

In [ ]:
correlation_matrix = hasilEkstrak.drop(columns=['Label','Filename']).corr()

threshold = 0.95 
columns = np.full((correlation_matrix.shape[0],), True, dtype=bool)

for i in range(correlation_matrix.shape[0]):
    for j in range(i+1, correlation_matrix.shape[0]):
        if abs(correlation_matrix.iloc[i,j]) >= threshold:
            if columns[j]: columns[j] = False

select = hasilEkstrak.drop(columns=['Label','Filename']).columns[columns]
x_new = hasilEkstrak[select]
y = hasilEkstrak['Label']

print("Grafik Heatmap Korelasi Antar Fitur (Yang Terseleksi):")
plt.figure(figsize=(14, 12))
sns.heatmap(x_new.corr(), annot=True, cmap='Blues', fmt=".2f")
plt.show()

---------

### 8. Seleksi Fitur Berbasis Korelasi (Feature Selection)
Pada tahap ini, kita melakukan penyaringan fitur untuk mengatasi masalah **multikolinearitas** (korelasi berlebihan antar variabel) yang dapat menyebabkan model menjadi bias atau bekerja terlalu lambat.

* **Logika Seleksi:** Kita menghitung matriks korelasi menggunakan `hasilEkstrak.corr()`. Jika dua fitur memiliki nilai korelasi absolut yang lebih besar atau sama dengan **0.95 (threshold)**, artinya kedua fitur tersebut membawa informasi yang hampir identik (*redundant*). Dalam kondisi ini, salah satu fitur akan dihapus untuk menjaga independensi data.
* **Manfaat:** * **Reduksi Dimensi:** Mengurangi jumlah fitur yang tidak perlu tanpa kehilangan informasi penting.
    * **Peningkatan Efisiensi:** Mempercepat waktu komputasi model *Machine Learning* karena variabel input menjadi lebih padat dan informatif.
    * **Pencegahan Overfitting:** Membantu model agar lebih fokus pada fitur-fitur yang benar-benar unik dalam membedakan tingkat kematangan pisang.
* **Visualisasi:** *Heatmap* ditampilkan untuk memberikan gambaran visual mengenai hubungan ketergantungan antar fitur yang telah lolos seleksi. Fitur dengan korelasi yang masih tinggi akan terlihat melalui gradasi warna, membantu kita memahami variabel mana yang paling berpengaruh dalam membedakan kelas kematangan.

---------


### STANDARISASI DATA

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x_new, y, test_size=0.2, random_state=42)

X_train_mean, X_train_std = X_train.mean(), X_train.std()
X_train = (X_train - X_train_mean) / X_train_std
X_test = (X_test - X_train_mean) / X_train_std

def generateClassificationReport(y_true, y_pred, title):
    print(f"------ {title} ------")
    print(classification_report(y_true, y_pred))
    print('Accuracy:', accuracy_score(y_true, y_pred), "\n")

rf = RandomForestClassifier(n_estimators=100, random_state=42)
svm = SVC(kernel='rbf', random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)

models = {'Random Forest': rf, 'SVM': svm, 'KNN': knn}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_test = model.predict(X_test)
    generateClassificationReport(y_test, y_pred_test, f"{name} (Testing Set)")

-----

### 9. Standardisasi Data dan Pelatihan Model
Setelah fitur diseleksi, langkah terakhir sebelum proses klasifikasi adalah standardisasi dan pelatihan model.

* **Standardisasi (Z-Score Normalization):**
  Data seringkali memiliki rentang nilai yang berbeda-beda (misalnya, nilai *Contrast* bisa ribuan, sementara *Energy* berada di rentang 0-1). Kita menggunakan rumus `(X - mean) / std` agar data memiliki rata-rata 0 dan standar deviasi 1. Standardisasi ini sangat penting terutama untuk model seperti **SVM** dan **KNN** yang sensitif terhadap skala data agar perhitungan jarak antar titik data menjadi adil.

* **Pelatihan Model (*Training*):**
  Kita menguji tiga algoritma populer:
    * **Random Forest:** Menggunakan metode *ensemble* (kumpulan pohon keputusan) untuk memberikan hasil klasifikasi yang stabil.
    * **SVM (*Support Vector Machine*):** Menggunakan kernel `rbf` untuk mencari pemisah terbaik antara kategori kematangan dalam ruang fitur dimensi tinggi.
    * **KNN (*K-Nearest Neighbors*):** Mengklasifikasikan data berdasarkan kesamaan fitur dengan 5 data terdekat (*k=5*).

* **Fungsi Evaluasi (`generateClassificationReport`):**
  Fungsi ini dipanggil secara otomatis untuk setiap model. Ia mencetak laporan klasifikasi yang mencakup metrik *Precision, Recall,* dan *F1-Score* untuk setiap kelas kematangan, serta akurasi keseluruhan. Hal ini memudahkan kita dalam membandingkan secara langsung performa ketiga model dalam satu alur eksekusi.

-----

### PELATIHAN MODEL

In [ ]:
knn = KNeighborsClassifier()
svm = SVC()
rf = RandomForestClassifier()

# Latih Model
knn.fit(X_train, y_train)
svm.fit(X_train, y_train)
rf.fit(X_train, y_train)

# Prediksi
y_pred_knn = knn.predict(X_test)
y_pred_svm = svm.predict(X_test)
y_pred_rf  = rf.predict(X_test)

print("Proses training dan prediksi selesai!")

--------

### 10. Pelatihan Model dan Inferensi (Model Training and Inference)
Tahap ini merupakan inti dari sistem klasifikasi, di mana model belajar untuk memetakan fitur-fitur tekstur yang telah diekstraksi ke label kategori kematangan pisang yang benar.

* **Inisialisasi Model:** Kita mendefinisikan tiga arsitektur model dengan karakteristik berbeda:
    * **K-Nearest Neighbors (KNN):** Menggunakan prinsip kedekatan (*proximity*) untuk mengklasifikasikan data berdasarkan kesamaan fitur dengan data latih terdekat. 
    * **Support Vector Machine (SVC):** Mencari bidang pemisah (*hyperplane*) optimal untuk memaksimalkan margin antara kelas kematangan pisang. 
    * **Random Forest (RF):** Menggunakan kumpulan pohon keputusan (*ensemble learning*) untuk mencapai akurasi yang lebih kuat dan tahan terhadap *overfitting*. 

[Image of random forest ensemble learning]


* **Proses Training (`.fit`):** Fungsi ini adalah tempat model "belajar". Data latih (`X_train`) dan target label (`y_train`) digunakan agar model dapat memahami pola numerik dari setiap kelas kematangan.

* **Proses Inferensi (`.predict`):** Setelah model terlatih, kita melakukan pengujian menggunakan data yang belum pernah dilihat sebelumnya (`X_test`). Hasil prediksi disimpan dalam variabel `y_pred` masing-masing model untuk kemudian dievaluasi kinerjanya pada tahap berikutnya.

----------

### EVALUASI MODEL

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y_test))
    
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(cmap=plt.cm.Blues, ax=ax)
    plt.title(title)
    plt.show()

print("Grafik Confusion Matrix untuk Ketiga Model:")
plot_confusion_matrix(y_test, rf.predict(X_test), "Random Forest Confusion Matrix")
plot_confusion_matrix(y_test, svm.predict(X_test), "SVM Confusion Matrix")
plot_confusion_matrix(y_test, knn.predict(X_test), "KNN Confusion Matrix")

-------

### 11. Evaluasi Model dengan Confusion Matrix
*Confusion Matrix* adalah alat evaluasi yang sangat informatif untuk memvisualisasikan performa klasifikasi secara detail. Berbeda dengan metrik akurasi yang bersifat global, matriks ini menunjukkan:

* **Diagonal Utama:** Menunjukkan jumlah prediksi yang benar (*True Positives*) untuk setiap kelas (Misalnya: Pisang Mentah terprediksi sebagai Mentah).
* **Elemen Luar Diagonal:** Menunjukkan kesalahan klasifikasi (*False Positives* dan *False Negatives*). Ini membantu kita mengidentifikasi kelas mana yang paling sulit dikenali oleh model.

Fungsi `plot_confusion_matrix` yang digunakan di atas memetakan hasil prediksi model (Random Forest, SVM, dan KNN) ke dalam bentuk *heatmap*. Penggunaan `cmap=plt.cm.Blues` mempermudah kita dalam mengidentifikasi tingkat kepercayaan model; semakin gelap warna pada diagonal, semakin tinggi performa model tersebut dalam mengenali kelas yang bersangkutan.

Analisis ini sangat berguna untuk melakukan *debugging* model: jika ada dua kelas yang sering tertukar, kita bisa memutuskan untuk menambahkan lebih banyak data latih pada kelas tersebut atau memperbaiki fitur ekstraksinya.

-------

### EKSEKUSI PRA-PEMROSESAN

In [ ]:
dataPreprocessed = []

for i in range(len(data)):
    img_current = data[i]
    
    if img_current is None:
        continue
        
    # Langsung gunakan data[i] tanpa resize
    img_array = np.array(img_current, dtype=np.uint8)
    
    # Langsung masuk ke proses preprocessing
    gray, median_blur, opening_result = prepro3_visual(img_array)
    
    # ---------------------------------------------------------
    # VISUALISASI
    # ---------------------------------------------------------
    fig, axes = plt.subplots(1, 4, figsize=(9, 4))
    
    img_rgb = cv.cvtColor(img_array, cv.COLOR_BGR2RGB) if len(img_array.shape) == 3 else img_array
        
    axes[0].imshow(img_rgb)
    axes[0].set_title(f"[{i}] Gambar Asli")
    axes[0].axis('off')
    
    axes[1].imshow(gray, cmap='gray')
    axes[1].set_title("1. Grayscale")
    axes[1].axis('off')
    
    axes[2].imshow(median_blur, cmap='gray')
    axes[2].set_title("2. Median Filter")
    axes[2].axis('off')
    
    axes[3].imshow(opening_result, cmap='gray')
    axes[3].set_title("3. Morfologi Opening")
    axes[3].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    dataPreprocessed.append(opening_result)

# PENTING: Jika ukuran gambar berbeda-beda, dataPreprocessed ini akan jadi 'list of arrays' 
# dan GLCM nanti mungkin akan error. 
dataPreprocessed = np.array(dataPreprocessed, dtype=object) 
print("Preprocessing selesai.")

--------

### 12. Eksekusi Pra-pemrosesan (Execution Pipeline)
Pada tahap ini, kita menerapkan fungsi pra-pemrosesan (`prepro3_visual`) yang telah didefinisikan sebelumnya ke seluruh dataset. Proses ini dilakukan secara iteratif untuk memastikan setiap citra mendapatkan perlakuan yang seragam.

* **Alur Transformasi:**
    1. **Gambar Asli:** Citra mentah yang dimuat dari *dataset*.
    2. **Grayscale (1):** Mengonversi citra berwarna menjadi intensitas cahaya. Langkah ini krusial untuk menyederhanakan data sebelum analisis tekstur dilakukan.
    3. **Median Filter (2):** Teknik penghalusan citra untuk menghilangkan *noise* atau *impulse noise* yang mungkin muncul saat pengambilan foto, menjaga tepi objek agar tetap tajam. 
    4. **Morfologi Opening (3):** Menggunakan operasi *erosion* diikuti *dilation* untuk membersihkan objek dari bercak-bercak kecil yang tidak relevan, sehingga fitur kulit pisang menjadi lebih dominan untuk dianalisis oleh matriks GLCM.

* **Manajemen Data:** Citra yang telah diproses disimpan ke dalam list `dataPreprocessed` dan dikonversi menjadi `numpy array` dengan `dtype=object`. Penggunaan `dtype=object` memberikan fleksibilitas untuk menampung gambar jika terdapat variasi dimensi, meskipun nantinya data ini akan diseragamkan sebelum masuk ke tahap ekstraksi fitur untuk menghindari error komputasi pada GLCM.

---------

### EVALUASI KUNATITATIF

In [ ]:
# --- EVALUASI MODEL ---
# Memastikan library sudah terhitung
acc_knn  = accuracy_score(y_test, y_pred_knn)
prec_knn = precision_score(y_test, y_pred_knn, average="weighted", zero_division=0)
rec_knn  = recall_score(y_test, y_pred_knn, average="weighted", zero_division=0)
f1_knn   = f1_score(y_test, y_pred_knn, average="weighted", zero_division=0)

acc_svm  = accuracy_score(y_test, y_pred_svm)
prec_svm = precision_score(y_test, y_pred_svm, average="weighted", zero_division=0)
rec_svm  = recall_score(y_test, y_pred_svm, average="weighted", zero_division=0)
f1_svm   = f1_score(y_test, y_pred_svm, average="weighted", zero_division=0)

acc_rf  = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf, average="weighted", zero_division=0)
rec_rf  = recall_score(y_test, y_pred_rf, average="weighted", zero_division=0)
f1_rf   = f1_score(y_test, y_pred_rf, average="weighted", zero_division=0)

# Membuat DataFrame Hasil
df_hasil = pd.DataFrame({
    "Classifier": ["KNN", "SVM", "Random Forest"],
    "Accuracy"  : [acc_knn,  acc_svm,  acc_rf],
    "Precision" : [prec_knn, prec_svm, prec_rf],
    "Recall"    : [rec_knn,  rec_svm,  rec_rf],
    "F1-Score"  : [f1_knn,   f1_svm,   f1_rf],
})
df_hasil = df_hasil.round(4)
df_hasil.to_csv("hasil_pp2.csv", index=False)

print("Tabel Perbandingan PP2:")
display(df_hasil)

----------

### 13. Evaluasi Kuantitatif Model
Pada tahap ini, kita mengukur efektivitas model dalam mengklasifikasikan tingkat kematangan pisang menggunakan metrik performa standar industri. Kita menggunakan parameter `average="weighted"` untuk memastikan metrik ini memperhitungkan distribusi kelas (penting jika jumlah data antar kelas tidak seimbang).

* **Metrik yang digunakan:**
    * **Accuracy:** Mengukur persentase prediksi yang benar secara keseluruhan.
    * **Precision:** Mengukur ketepatan model—seberapa sering model benar ketika memprediksi kelas tertentu.
    * **Recall:** Mengukur kemampuan model untuk menemukan semua sampel yang relevan dalam satu kelas.
    * **F1-Score:** Rata-rata harmonik antara *Precision* dan *Recall*; ini adalah metrik paling seimbang untuk menilai performa model saat *Precision* dan *Recall* mungkin berbeda jauh.

* **Penyusunan Hasil:** Semua metrik dihitung untuk ketiga model (KNN, SVM, Random Forest) dan disatukan ke dalam `pandas.DataFrame`. Hasil ini kemudian disimpan ke dalam format `hasil_pp2.csv` untuk keperluan dokumentasi hasil eksperimen dan pelaporan lebih lanjut.

----------

### VISUALISASI PERBANDINGAN

In [ ]:
# --- VISUALISASI ---
metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
x       = np.arange(len(metrics))
width   = 0.25
clr     = ["#89b4fa", "#f38ba8", "#a6e3a1"]

fig, ax = plt.subplots(figsize=(10, 5))
for i, (clf, color) in enumerate(zip(["KNN", "SVM", "Random Forest"], clr)):
    vals = df_hasil[df_hasil["Classifier"] == clf][metrics].values.flatten()
    bars = ax.bar(x + i * width, vals, width, label=clf, color=color, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{v:.2f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_title("Perbandingan KNN vs SVM vs Random Forest — PP2", fontweight="bold")
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("grafik_pp2.png", dpi=120, bbox_inches="tight")
plt.show()

-------------

### 14. Visualisasi Perbandingan Performa (Model Performance Comparison)
Setelah metrik performa dihitung, tahap terakhir adalah memvisualisasikan hasil tersebut dalam bentuk *grouped bar chart*. Visualisasi ini bertujuan untuk:

* **Analisis Komparatif:** Memudahkan kita untuk membandingkan secara langsung nilai *Accuracy, Precision, Recall,* dan *F1-Score* antara ketiga model (KNN, SVM, dan Random Forest). 
* **Identifikasi Kekuatan Model:** Warna yang berbeda pada setiap *bar* (menggunakan palet warna yang kontras) membantu kita melihat secara cepat model mana yang mendominasi di setiap metrik evaluasi.
* **Presisi Pelaporan:** Penggunaan `ax.text` untuk menampilkan angka tepat di atas setiap *bar* memastikan bahwa data performa dapat dibaca dengan presisi tinggi oleh pembaca laporan, tanpa harus merujuk kembali ke tabel data.

Visualisasi ini disimpan ke dalam file `grafik_pp2.png` untuk kemudian dapat disisipkan langsung ke dalam dokumen laporan akhir atau slide presentasi praktikum.

-------------

### RINGKASAN MODEL

In [ ]:
# --- HASIL TERBAIK ---
best = df_hasil.loc[df_hasil["Accuracy"].idxmax()]

print("=" * 40)
print("   HASIL TERBAIK PP2")
print("=" * 40)
print(f"   Classifier : {best['Classifier']}")
print(f"   Accuracy   : {best['Accuracy']:.4f}")
print(f"   Precision  : {best['Precision']:.4f}")
print(f"   Recall     : {best['Recall']:.4f}")
print(f"   F1-Score   : {best['F1-Score']:.4f}")
print("=" * 40)

------------

### 15. Ringkasan Model Terbaik (Best Model Selection)
Setelah melakukan pengujian terhadap berbagai algoritma (KNN, SVM, dan Random Forest), tahap terakhir adalah melakukan seleksi otomatis untuk menentukan model mana yang memiliki performa paling unggul.

* **Logika Pemilihan:** Kita menggunakan fungsi `idxmax()` pada kolom `Accuracy` untuk menemukan indeks model dengan performa akurasi tertinggi dalam `df_hasil`.
* **Tujuan:** Ringkasan ini berfungsi untuk memberikan kesimpulan yang jelas kepada pembaca mengenai model mana yang paling direkomendasikan untuk digunakan dalam aplikasi klasifikasi kematangan pisang berdasarkan dataset yang diuji.
* **Transparansi Hasil:** Dengan menampilkan metrik lengkap (*Accuracy, Precision, Recall,* dan *F1-Score*) dari model terpilih secara eksplisit, kita memastikan bahwa pemilihan model tidak hanya berdasarkan satu parameter saja, melainkan representasi performa keseluruhan yang akurat.

Laporan ringkasan ini menjadi titik akhir dalam alur kerja klasifikasi kita, yang membuktikan bahwa integrasi antara pra-pemrosesan citra, ekstraksi fitur tekstur GLCM, dan pemodelan *Machine Learning* mampu menghasilkan sistem klasifikasi yang terukur dan efisien.

------------